In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.model_selection import cross_val_score
warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

In [2]:
df = pd.read_csv(r'C:\Users\USER\Desktop\New folder (3)\data anlaysis pb\assignmenttwo\Train_data.csv')
dfd = pd.read_csv(r'C:\Users\USER\Desktop\New folder (3)\data anlaysis pb\assignmenttwo\Variable Description.csv')

In [3]:
#Basic EDA
df.head()

,Customer Id,YearOfObservation,Insured_Period,Residential,Building_Painted,Building_Fenced,Garden,Settlement,Building Dimension,Building_Type,Date_of_Occupancy,NumberOfWindows,Geo_Code,Claim
0,H14663,2013,1.0,0,N,V,V,U,290.0,1,1960.0,.,1053,0
1,H2037,2015,1.0,0,V,N,O,R,490.0,1,1850.0,4,1053,0
2,H3802,2014,1.0,0,N,V,V,U,595.0,1,1960.0,.,1053,0
3,H3834,2013,1.0,0,V,V,V,U,2840.0,1,1960.0,.,1053,0
4,H5053,2014,1.0,0,V,N,O,R,680.0,1,1800.0,3,1053,0


In [4]:
#Basic EDA
df.describe()

,YearOfObservation,Insured_Period,Residential,Building Dimension,Building_Type,Date_of_Occupancy,Claim
count,7160.000000,7160.000000,7160.000000,7054.000000,7160.000000,6652.000000,7160.000000
mean,2013.669553,0.909758,0.305447,1883.727530,2.186034,1964.456404,0.228212
std,1.383769,0.239756,0.460629,2278.157745,0.940632,36.002014,0.419709
min,2012.000000,0.000000,0.000000,1.000000,1.000000,1545.000000,0.000000
25%,2012.000000,0.997268,0.000000,528.000000,2.000000,1960.000000,0.000000
50%,2013.000000,1.000000,0.000000,1083.000000,2.000000,1970.000000,0.000000
75%,2015.000000,1.000000,1.000000,2289.750000,3.000000,1980.000000,0.000000
max,2016.000000,1.000000,1.000000,20940.000000,4.000000,2016.000000,1.000000


In [5]:
#Basic EDA
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7160 entries, 0 to 7159
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Customer Id         7160 non-null   object 
 1   YearOfObservation   7160 non-null   int64  
 2   Insured_Period      7160 non-null   float64
 3   Residential         7160 non-null   int64  
 4   Building_Painted    7160 non-null   object 
 5   Building_Fenced     7160 non-null   object 
 6   Garden              7153 non-null   object 
 7   Settlement          7160 non-null   object 
 8   Building Dimension  7054 non-null   float64
 9   Building_Type       7160 non-null   int64  
 10  Date_of_Occupancy   6652 non-null   float64
 11  NumberOfWindows     7160 non-null   object 
 12  Geo_Code            7058 non-null   object 
 13  Claim               7160 non-null   int64  
dtypes: float64(3), int64(4), object(7)
memory usage: 783.3+ KB


In [6]:
#Dropping columns thta doesnt aid in building our model("Customer Id","Geo_Code")
df = df.drop(columns=['Customer Id','Geo_Code'])

In [7]:
df['Date_of_Occupancy'].unique().sum()

np.float64(nan)

In [8]:
df['Building_Painted'] = df['Building_Painted'].replace('N',0)
df['Building_Painted'] = df['Building_Painted'].replace('V',1)
df['Building_Fenced'] =  df['Building_Fenced'].replace('N',0)
df['Building_Fenced'] = df['Building_Fenced'].replace('V',1)
df['Garden'] = df['Garden'].replace('V',1)
df['Garden'] = df['Garden'].replace('O',0)
df['Settlement'] = df['Settlement'].replace('R',0)
df['Settlement'] = df['Settlement'].replace('U',1)

In [9]:
df['NumberOfWindows'].unique()
df['NumberOfWindows'] = df['NumberOfWindows'].replace('   .', np.nan)
df['NumberOfWindows'] = df['NumberOfWindows'].replace('>=10',10)

In [10]:
df['NumberOfWindows'] = df['NumberOfWindows'].isna()
df['NumberOfWindows'].sum()

np.int64(3551)

In [11]:
#to define x and also drop claim which is the output and input its output into y
x = df.drop(columns = 'Claim', axis = 1)
y = df['Claim']

In [12]:
#to find the amount of na in the building dimension column
x['Building Dimension'] = x['Building Dimension'].fillna(x['Building Dimension'].mean())
x['Garden'] = x['Garden'].fillna(x['Garden'].mode())
x['Date_of_Occupancy'] = x['Date_of_Occupancy'].fillna(x['Date_of_Occupancy'].mean())

#x['Garden'].unique()#to get the unique col in th garden column

In [13]:
print(f'The amount of data types in  both Garden and building dimension are \nGarden: {x['Garden'].unique()} \nBuilding Dimension: {x['Building Dimension'].isna().sum()}')

The amount of data types in  both Garden and building dimension are 
Garden: [ 1.  0. nan] 
Building Dimension: 0


In [14]:

x['Garden'] = x['Garden'].fillna(x['Garden'].mean())
x.isna().sum()

YearOfObservation     0
Insured_Period        0
Residential           0
Building_Painted      0
Building_Fenced       0
Garden                0
Settlement            0
Building Dimension    0
Building_Type         0
Date_of_Occupancy     0
NumberOfWindows       0
dtype: int64

In [15]:
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size = 0.2, random_state = 42)

In [16]:
scaler = StandardScaler()
x_train_standardized = scaler.fit_transform(x_train)
x_test_standardized = scaler.transform(x_test)
print(x_train_standardized)


[[ 1.68333     0.36173668  1.52019165 ...  1.945864   -0.12549207
   1.01194282]
 [ 0.96004486  0.37325912 -0.6578118  ... -0.19059546 -0.12549207
   1.01194282]
 [-0.48652543  0.37325912 -0.6578118  ...  0.87763427  0.16116871
  -0.98819813]
 ...
 [ 0.23675971  0.37325912  1.52019165 ...  1.945864   -1.73079245
  -0.98819813]
 [-1.20981057  0.37325912  1.52019165 ... -0.19059546 -0.12549207
   1.01194282]
 [-1.20981057  0.37325912  1.52019165 ... -1.25882518 -0.41215286
  -0.98819813]]


In [17]:
print(x_test_standardized)

[[-1.20981057  0.37325912 -0.6578118  ... -0.19059546 -0.12549207
   1.01194282]
 [-0.48652543  0.37325912 -0.6578118  ...  0.87763427  0.44782949
   1.01194282]
 [-1.20981057 -1.5048997   1.52019165 ...  1.945864    0.44782949
   1.01194282]
 ...
 [ 0.23675971  0.37325912 -0.6578118  ... -0.19059546  1.10714929
  -0.98819813]
 [ 0.23675971  0.37325912 -0.6578118  ... -1.25882518  0.67715812
   1.01194282]
 [-1.20981057  0.37325912 -0.6578118  ... -0.19059546  1.30781184
  -0.98819813]]


In [18]:
# lr = LinearRegression()
# #training the model
# lr.fit(x_train, y_train)

In [19]:
#Applying the model to make a prediction
# y_lr_train_pred = lr.predict(x_train)
# y_lr_test_pred = lr.predict(x_test)

In [20]:
# #evaluating model perfomance
# lr_train_mse = mean_squared_error(y_train,y_lr_train_pred)
# lr_train_r2 = r2_score(y_train,y_lr_train_pred)
#
# lr_test_mse = mean_squared_error(y_test,y_lr_test_pred)
# lr_test_r2 = r2_score(y_test,y_lr_test_pred)

In [21]:
# lr_train_r2

In [22]:
# lr_results = pd.DataFrame(['Linear Regresson',lr_train_mse,lr_train_r2,lr_test_mse,lr_test_r2]).transpose()

In [23]:
# lr_results.columns = ['Method','Training MSE','Training RSquare','Test MSE','Test RSquare']

In [24]:
# lr_results

In [34]:
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

#Initialize model
rf = RandomForestClassifier(
    n_estimators=400,
    class_weight="balanced",  # helps with imbalanced claims
    random_state=42
)

#Train model
rf.fit(x_train, y_train)

#Predict probabilities (for AUC)
y_prob = rf.predict_proba(x_test)[:, 1]

# Predict classes
y_pred = rf.predict(x_test)

AUC_rf = roc_auc_score(y_test, y_prob)

# Evaluate
print("AUC:", roc_auc_score(y_test, y_prob))#AUC = Area under the roc curve
print(classification_report(y_test, y_pred))
#Accuracy
accuracy_rf = accuracy_score(y_test, y_pred) * 100
print(f"Random Forest Accuracy: {accuracy_rf:.2f}%")
#precision
prec_rf = precision_score(y_test, y_pred)
#recall score
rec_rf = recall_score(y_test, y_pred)
#f1 score
f1_rf = f1_score(y_test, y_pred)

print('Rf precision: ',prec_rf)
print('Rf recall: ',rec_rf)
print('Rf f1_score: ',f1_rf)

AUC: 0.6603296139960516
              precision    recall  f1-score   support

           0       0.80      0.91      0.85      1098
           1       0.47      0.27      0.34       334

    accuracy                           0.76      1432
   macro avg       0.64      0.59      0.60      1432
weighted avg       0.73      0.76      0.73      1432

Random Forest Accuracy: 75.91%
Rf precision:  0.4708994708994709
Rf recall:  0.26646706586826346
Rf f1_score:  0.3403441682600382


In [35]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

lr = LogisticRegression(max_iter=1000,random_state=42)
#Trainig the model using Logistic regression
lr.fit(x_train, y_train)


y_prob = lr.predict_proba(x_test)[:, 1]
y_pred_lr = lr.predict(x_test)
#AUC
AUC_lr = roc_auc_score(y_test, y_prob)

print("AUC:", roc_auc_score(y_test, y_prob))#AUC = Area under the roc curve
#Accuracy
accuracy_lr = accuracy_score(y_test, y_pred_lr) * 100
print(f"Logistic Regression Accuracy: {accuracy_lr:.2f}%")
#precison
prec_lr = precision_score(y_test, y_pred_lr)
#recall score
rec_lr = recall_score(y_test, y_pred_lr)
#f1 score
f1_lr = f1_score(y_test, y_pred_lr)
print('lr precision: ',prec_lr)
print('lr Recall: ',rec_lr)
print('lr f1_score: ',f1_lr)

AUC: 0.7141945616962795
Logistic Regression Accuracy: 78.14%
lr precision:  0.64
lr Recall:  0.1437125748502994
lr f1_score:  0.23471882640586797


In [50]:
model_table = pd.DataFrame(['Logistic Regression', accuracy_lr,AUC_lr,rec_lr,f1_lr,prec_lr]).transpose()
model_table.columns = ['Method','Accuracy','AUC','Recall','F1','Prediction']

In [73]:
# rfp = pd.DataFrame({
#     'Method':'Random Forest Classifier',
#     'Accuracy': accuracy_rf,
#     'AUC': AUC_rf,
#     'Recall': rec_rf,
#     'F1':f1_rf,
#     'Prediction':prec_rf
# })
rpf = pd.DataFrame(["Random Forest Classifier", accuracy_rf, AUC_rf, rec_rf, f1_rf,prec_rf]).transpose()
rpf.columns = ['Method','Accuracy','AUC','Recall','F1','Prediction']

In [74]:
rpf

,Method,Accuracy,AUC,Recall,F1,Prediction
0,Random Forest Classifier,75.907821,0.66033,0.266467,0.340344,0.470899


In [79]:
# model_table = model_table.concat(rpf,ignore_index=True )
model_table= pd.concat([model_table,rpf],ignore_index=True)


In [84]:
model_table = model_table.reset_index(drop=True)
model_table

,Method,Accuracy,AUC,Recall,F1,Prediction
0,Logistic Regression,78.142458,0.714195,0.143713,0.234719,0.64
1,Random Forest Classifier,75.907821,0.66033,0.266467,0.340344,0.470899


In [86]:
model_table.to_csv('irenikase_samuel_T.csv', index = False)

In [91]:
model_table.to_html('C:/Users/USER/Downloads/irenikase_samuel_T.html', index = False)

In [131]:
#to select the data type that are not obj  ie. str
num_col = [col for col in x.columns if x[col].dtype != 'object']

In [105]:
#to select the data type that are object ie. str
cat_col = [col for col in x.columns if x[col].dtype == 'object']